# Multi-Node Training on SageMaker Training job

In [1]:
# ## Update sagemaker python sdk version
!pip install -U sagemaker transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 157.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 164.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 166.3 MB/s eta 0:00:00
  Attempting uninstall: sagemaker━╸━━━━━━━━━━━━━ 4/6 [transformers]ub]
    Found existing installation: sagemaker 2.248.2━━━━━━━━━━━━ 4/6 [transformers]
    Uninstalling sagemaker-2.248.2:━━━━━━╺━━━━━━ 5/6 [sagemaker]
      Successfully uninstalled sagemaker-2.248.2━━━━━━ 5/6 [sagemaker]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [sagemaker]/6 [sagemaker]


## Set model, Code and data

In [2]:
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
sagemaker_default_bucket = sess.default_bucket()
region = sess.boto_session.region_name
print("sagemaker_default_bucket:", sagemaker_default_bucket)
print("sagemaker_region:", region)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker_default_bucket: sagemaker-us-east-1-596899493901
sagemaker_region: us-east-1


## upload base models to s3

In [3]:
!pip install huggingface_hub

In [ ]:
# Code language: python
from huggingface_hub import snapshot_download
from pathlib import Path

# model_name = "Qwen/Qwen2.5-7B-Instruct"
model_name = "Qwen/Qwen3-14B"
model_file =model_name.split("/")[-1]
local_cache_path = Path(f"../{model_file}")
local_cache_path.mkdir(exist_ok=True)

# Only download pytorch checkpoint files
allow_patterns = ["*"]

model_download_path = snapshot_download(
    repo_id=model_name,
    cache_dir=local_cache_path,
    allow_patterns=allow_patterns,
)
model_snapshot_path = list(local_cache_path.glob("**/snapshots/*"))[0]

In [12]:
!aws s3 cp {model_snapshot_path} s3://{sagemaker_default_bucket}/Foundation-Models/{model_file}  --recursive

upload: Qwen3-14B/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/.gitattributes to s3://sagemaker-us-east-1-596899493901/Foundation-Models/Qwen3-14B_1/.gitattributes
upload: Qwen3-14B/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/LICENSE to s3://sagemaker-us-east-1-596899493901/Foundation-Models/Qwen3-14B_1/LICENSE
upload: Qwen3-14B/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/config.json to s3://sagemaker-us-east-1-596899493901/Foundation-Models/Qwen3-14B_1/config.json
upload: Qwen3-14B/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/generation_config.json to s3://sagemaker-us-east-1-596899493901/Foundation-Models/Qwen3-14B_1/generation_config.json
upload: Qwen3-14B/models--Qwen--Qwen3-14B/snapshots/40c069824f4251a91eefaf281ebe4c544efd3e18/README.md to s3://sagemaker-us-east-1-596899493901/Foundation-Models/Qwen3-14B_1/README.md
upload: Qwen3-14B/models--Qwen--Qwen3-14B/sn

## Setup for wandb

In [13]:
!pip install wandb

  Using cached gitpython-3.1.45-py3-none-any.whl.metadata (13 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 129.3 MB/s eta 0:00:00
Using cached gitpython-3.1.45-py3-none-any.whl (208 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached smmap-5.0.2-py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [wandb]32m4/5 [wandb]-sdk]


In [14]:
import wandb
wandb.login()

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ec2-user/.netrc
wandb: Currently logged in as: 407383787 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Submit Training job

In [28]:
data_s3_uri = "s3://finetune-data-dxq/example_data.json"
!aws s3 cp example_data.json {s3_uri}

upload: ./example_data.json to s3://finetune-data-dxq/example_data.json


In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.pytorch import PyTorch
from datetime import datetime


instance_count = 1
# instance_type = 'ml.p5en.48xlarge'
# instance_type = 'ml.g5.48xlarge'  ## 8*24G
# instance_type = 'ml.g6e.48xlarge'
instance_type = 'ml.g6e.48xlarge'
max_time = 86400 # 24小时

wandb.sagemaker_auth(path="./")


model_s3_checkpoint_path = f"s3://{sagemaker_default_bucket}/finetuned_model/{model_file}_checkpoints/"
environment = {
    'NODE_NUMBER':str(instance_count),
    'MODEL_S3_PATH': f's3://{sagemaker_default_bucket}/Foundation-Models/{model_file}', # source model files
    'MODEL_LOCAL_PATH': '/tmp/base_model'
}
prefix = "sft-2"
estimator = PyTorch(entry_point='entry.py',
                            source_dir='./',
                            role=role,
                            environment=environment,
                            framework_version='2.7.1',
                            base_job_name=prefix,
                            py_version='py312',
                            checkpoint_s3_uri=model_s3_checkpoint_path,
                            script_mode=True,
                            instance_count=instance_count,
                            instance_type=instance_type,
                            keep_alive_period_in_seconds=3600,
                            max_run=max_time)

# data in channel will be automatically copied to each node - /opt/ml/input/data/train1
# 可以需要替换成自己的数据集，拉取的训练机器会将数据下载到  /opt/ml/input/data/train 目录下
input_channel = {'train': data_s3_uri}
# ssh_wrapper = SSHEstimatorWrapper.create(estimator, connection_wait_time_seconds=600)
estimator.fit(input_channel)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: sft-2-2025-08-21-09-30-25-631


2025-08-21 09:30:29 Starting - Starting the training job
2025-08-21 09:30:29 Pending - Training job waiting for capacity..........................................
2025-08-21 09:37:20 Pending - Preparing the instances for training............
2025-08-21 09:39:15 Downloading - Downloading input data..................
2025-08-21 09:42:27 Downloading - Downloading the training image.........
2025-08-21 09:43:58 Training - Training image download completed. Training in progress....bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
Skipping CUDA compat setup as package not found
2025-08-21 09:44:27,975 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-08-21 09:44:28,053 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-08-21 09:44:28,061 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-08-21 09:44:2

In [18]:
!aws s3 ls {model_s3_checkpoint_path}

                           PRE v0-20250821-083417/
